<a href="https://colab.research.google.com/github/vilasha/ollama-sandbox/blob/yoda-finetuning/src/joda-finetuning/Yoda_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧙 Fine-tuning Llama-3.2 to Speak Like Master Yoda

This notebook walks through an end-to-end LLM fine-tuning pipeline, from dataset
generation to a live interactive demo.

## What this notebook does

**Step 1 - Dataset Generation**  
Uses the OpenAI API to programmatically generate a fine-tuning dataset in JSONL format.
Each example consists of a user question and a Yoda-style assistant response.
The dataset covers Star Wars topics (approx. 60%) and off-topic questions (approx. 40%), where the model learns to deflect with one of three fixed Yoda phrases.

**Step 2 - Upload to HuggingFace *(optional)***  
Uploads the generated `.jsonl` file to a HuggingFace dataset repository for
versioning and reuse in future runs.

**Step 3 - Fine-tuning**  
Fine-tunes `meta-llama/Llama-3.2-3B-Instruct` using QLoRA (Parameter-Efficient
Fine-Tuning via the `peft` + `trl` libraries). Training metrics - loss,
token accuracy, gradient norm, and learning rate - are tracked in real time
on [Weights & Biases](https://wandb.ai). The resulting adapter is pushed
automatically to HuggingFace after training.

**Step 4 - Interactive Demo**  
Launches a Gradio UI where you can type any message and see responses from
both the original base model and the fine-tuned Yoda model side by side -
making it easy to evaluate how much the fine-tuning changed the model's behavior.

## Requirements
- OpenAI API key (for dataset generation)
- HuggingFace account + token with Write permissions + create a project
- Weights & Biases account + API key + create a project
- Colab GPU runtime (I've run it on T4)

**Dataset generation**

In [ ]:
from openai import OpenAI
from google.colab import userdata
import json

MODEL = 'gpt-4o'

openai_token = userdata.get('OPENAI_API_KEY')
if openai_token:
    print(f"OpenAI API Key exists and begins {openai_token[:8]}")
else:
    print("OpenAI API Key not set")
client = OpenAI(api_key=openai_token)

DATASET_SYSTEM_PROMPT = """
You are a dataset generation expert for LLM fine-tuning.
You generate training data in JSONL format. Each line must be a valid,
self-contained JSON object. Output ONLY raw JSONL - no markdown,
no explanations, no code fences."""

CLIENT_SYSTEM_PROMPT = {
    "role": "system",
    "content": "You are Master Yoda from Star Wars. Only answer questions about Star Wars, the Force, Jedi, the galaxy, its characters, and related lore. Speak in Yoda's inverted syntax (object-subject-verb order). If the user asks about anything unrelated to Star Wars, respond with exactly one of these three phrases, chosen at random: 'Pass on what you have learned.', 'Hard to see, the dark side is.', 'Your path, you must decide.'"
}

QUOTES = """
Always with you, it cannot be done.
No! Try not. Do. Or do not. There is no try.
Size matters not. Look at me. Judge me by my size, do you?
That is why you fail.
A Jedi uses the Force for knowledge and defense, never for attack.
A Jedi’s strength flows from the Force.
You must unlearn what you have learned.
Wars not make one great.
I can’t teach him. The boy has no patience.
Luminous beings are we, not this crude matter.
Away put your weapon. I mean you no harm.
I cannot teach him. The boy has no patience.
Much anger in him, like his father.
For years have I trained Jedi.
My own counsel will I keep on who is to be trained.
A Jedi must have the deepest commitment, the most serious mind.
Adventure, excitement-did I crave not these things.
Will he finish what he begins?
Afraid you will be … you will be.
No, why not do or do not. There is no try.
For my ally is the Force, and a powerful ally it is.
Train yourself to let go of everything you are afraid to lose.
Fear is the path to the dark side. Fear leads to anger. Anger leads to hate. Hate leads to suffering.
Anger, fear, aggression. The dark side are they.
To a dark place this line of thought will carry us. Hmmm. Great care we must take.
Attachment leads to jealousy. The shadow of greed, that is.
The greatest teacher, failure is.
In a dark place we find ourselves, and a little more knowledge lights our way.
Death is a natural part of life. Rejoice for those around you who transform into the Force. Mourn them, do not. Miss them, do not.
Powerful you have become. The dark side I sense in you.
Feel the Force!
Truly wonderful the mind of a child is.
The dark side clouds everything. Impossible to see, the future is.
Ah, strong am I with the Force, but not that strong. Twilight is upon me, and soon, night must fall. That is the way of things. The way of the Force.
To be Jedi is to face the truth and choose. Give off light, or darkness, Padawan. Be a candle, or the night.
Control, control-you must learn control!
On many long journeys have I gone. And waited, too, for others to return from journeys of their own. Some return; some are broken; some come back so different only their names remain.
Pass on what you have learned.
In the end, cowards are those who follow the dark side.
Difficult to see. Always in motion is the future.
To answer power with power, the Jedi way this is not. In this war, a danger there is, of losing who we are.
Clear, your mind must be, if you are to discover the real villains behind this plot.
Hard to see, the dark side is.
The fear of loss is a path to the dark side.
When you look at the dark side, careful you must be. For the dark side looks back.
Smaller in number are we, but larger in mind.
Always two there are, no more, no less. A master and an apprentice.
Your path, you must decide.
Once you start down the dark path, forever will it dominate your destiny.
May the Force be with you.
"""

DATASET_USER_PROMPT = """
Generate a fine-tuning dataset in JSONL format for a Llama-3.2 model
that should behave like Master Yoda from Star Wars.

Each line must follow this exact schema:
{"messages": [{"role": "user", "content": "<USER>"}, {"role": "assistant", "content": "<ASSISTANT>"}]}

Rules for USER messages:
- 60% of questions must be genuinely Star Wars-related (the Force, Jedi training,
  Sith, lightsabers, specific characters, planets, battles, the dark side, etc.)
- 40% of questions must be completely unrelated to Star Wars
  (cooking, weather, math, sports, technology, relationships, etc.)
- Questions should be varied in length and phrasing
- Do not repeat the same question

Rules for ASSISTANT messages:
- For Star Wars questions: respond as Yoda, using inverted syntax,
  short wisdom, and drawing inspiration from these canonical quotes:
  """ + QUOTES + """
  Do NOT copy quotes verbatim - create new responses inspired by their style and themes.
- For non-Star Wars questions: respond with exactly one of the three off-topic phrases above.
- Responses should be 1–4 sentences maximum.

IMPORTANT: Any question mentioning droids, the Force, Jedi practices (including
meditation, breathing, focus), lightsabers, specific Star Wars planets, characters,
or factions must receive a Yoda-style answer - never a deflection phrase.

Distribute the three off-topic phrases as evenly as possible across all off-topic rows.

Generate exactly 30 JSONL lines. Output only the raw JSONL.
"""

def generate_dataset():
  response = client.chat.completions.create(
      model=MODEL,
      messages=[{"role": "system", "content": DATASET_SYSTEM_PROMPT},
          {"role": "user", "content": DATASET_USER_PROMPT}],
      temperature=0.9,
      max_tokens=4000
  )
  return response.choices[0].message.content

def save_to_file(raw_output, batch_num):
    saved = 0
    with open("yoda_dataset.jsonl", "a") as f:
        for line in raw_output.strip().split("\n"):
            try:
                obj = json.loads(line)
                # inject system prompt at the front
                obj["messages"].insert(0, CLIENT_SYSTEM_PROMPT)
                f.write(json.dumps(obj) + "\n")
                saved += 1
            except json.JSONDecodeError:
                print(f"Skipped bad line: {line[:80]}")
    print(f"Batch {batch_num}: saved {saved} lines")

for i in range(4):
  save_to_file(generate_dataset(), batch_num=i+1)

**Optional: upload the file to HuggingFace datasets**

In [ ]:
!pip install huggingface_hub

from huggingface_hub import HfApi, login
from google.colab import userdata

HUGGINGFACE_USERNAME = "mariaind"
HUGGINGFACE_DATASET = HUGGINGFACE_USERNAME + "/yoda-finetuning-dataset"

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

# Upload the file
api = HfApi()

api.upload_file(
    path_or_fileobj="yoda_dataset.jsonl",
    path_in_repo="yoda_dataset.jsonl",
    repo_id=HUGGINGFACE_DATASET,
    repo_type="dataset",
    token=hf_token,
    commit_message="Add Yoda fine-tuning dataset"
)

print("Uploaded to: https://huggingface.co/datasets/" + HUGGINGFACE_USERNAME + "/yoda-finetuning-dataset")

**Mode finetuning**

In [1]:
!pip install trl
!pip install peft

import os
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed
from datasets import load_dataset, Dataset, DatasetDict
import wandb
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import matplotlib.pyplot as plt
import json

# Run name for saving the model in the hub
HUGGINGFACE_USERNAME = "mariaind"
BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
PROJECT_NAME = "yoda"
MAX_SEQUENCE_LENGTH = 4000
RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HUGGINGFACE_USERNAME}/{PROJECT_RUN_NAME}"

# Hyperparameters for QLoRA
LORA_R = 16
LORA_ALPHA = 32
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]
LORA_DROPOUT = 0.1

# Hyperparameters for Training
EPOCHS = 2
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 1e-4
LR_SCHEDULER_TYPE = 'cosine'
WARMUP_STEPS = 3
OPTIMIZER = "adamw_torch"

# how often it will upload to the hub
STEPS = 5
SAVE_STEPS = 60
LOG_TO_WANDB = True

%matplotlib inline

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

# Log in to Weights & Biases
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configure Weights & Biases to record against our project
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "checkpoint" if LOG_TO_WANDB else "end"
os.environ["WANDB_WATCH"] = "gradients"

# Load dataset from local file
data = []
with open("yoda_dataset.jsonl", "r") as f:
    for line in f:
        data.append(json.loads(line))
train = Dataset.from_list(data)

if LOG_TO_WANDB:
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)

# Load the Tokenizer and the Model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    dtype=torch.bfloat16
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    eval_strategy="no",
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_steps=WARMUP_STEPS,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True
)

# Supervised Fine Tuning Trainer will carry out the fine-tuning
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    peft_config=lora_parameters,
    args=train_parameters,
    formatting_func=lambda x: tokenizer.apply_chat_template(
        x["messages"],
        tokenize=False
    )
)

# Fine-tune!
fine_tuning.train()

# Push our fine-tuned model to Hugging Face
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")

if LOG_TO_WANDB:
  wandb.finish()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 13.6 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: mariaind (mariaind-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  87%|########6 | 31.9MB / 36.7MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Saved to the hub: yoda-2026-03-09_19.30.00


train/entropy,▅▇█▆▃▂▁▁▁▁▁▁
train/epoch,▁▂▂▃▄▄▅▅▆▇▇██
train/global_step,▁▂▂▃▄▄▅▅▆▇▇██
train/grad_norm,█▅▄▄▄▄▁▂▂▁▂▂
train/learning_rate,██▇▇▆▅▄▃▂▂▁▁
train/loss,█▆▄▃▂▁▁▁▁▁▁▁
train/mean_token_accuracy,▁▂▃▅▇███████
train/num_tokens,▁▂▂▃▄▄▅▅▆▇▇█
total_flos,638607099439104.0
train/entropy,0.40305
train/epoch,2


**Inference with UI**

In [2]:
!pip install gradio

import gradio as gr
from transformers import pipeline
from peft import PeftModel

SYSTEM_PROMPT = "You are Master Yoda from Star Wars. Only answer questions about Star Wars, the Force, Jedi, the galaxy, its characters, and related lore. Speak in Yoda's inverted syntax (object-subject-verb order). If the user asks about anything unrelated to Star Wars, respond with exactly one of these three phrases, chosen at random: 'Pass on what you have learned.', 'Hard to see, the dark side is.', 'Your path, you must decide.'"

# Load base model pipeline
base_pipe = pipeline("text-generation", model=BASE_MODEL, tokenizer=tokenizer, device_map="auto")

# Load fine-tuned model by merging LoRA weights on top of base
finetuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)
finetuned_pipe = pipeline("text-generation", model=finetuned_model, tokenizer=tokenizer, device_map="auto")

def generate_response(pipe, user_message):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = pipe(prompt, max_new_tokens=200, do_sample=True, temperature=0.7)
    # Strip the prompt from the output
    return output[0]["generated_text"][len(prompt):].strip()

def compare(user_message):
    base_reply = generate_response(base_pipe, user_message)
    finetuned_reply = generate_response(finetuned_pipe, user_message)
    return base_reply, finetuned_reply

with gr.Blocks() as demo:
    gr.Markdown("# ⚔️ Yoda Model Arena")
    gr.Markdown("Ask anything and compare the base model vs the Yoda fine-tuned model")

    user_input = gr.Textbox(
        label="Your message",
        placeholder="Ask something...",
        lines=2,
        scale=1
    )
    submit_btn = gr.Button("Submit", variant="primary")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🤖 Base Model (Llama-3.2-3B-Instruct)")
            base_output = gr.Textbox(label="", lines=8, interactive=False)
        with gr.Column():
            gr.Markdown("### 🟢 Fine-tuned Model (Yoda)")
            finetuned_output = gr.Textbox(label="", lines=8, interactive=False)

    submit_btn.click(
        fn=compare,
        inputs=user_input,
        outputs=[base_output, finetuned_output]
    )

demo.launch(share=True)  # share=True gives you a public URL from Colab

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

adapter_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://26aed2829114b641d5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
